# LicitAI: memoria de calculo financeiro

Data-base: 2026-10-01. Simulacao, nao dados de vendas reais. Valores em BRL, salvo indicacao de USD. Executavel com Python 3 e biblioteca padrao. Leitura principal: briefing-validacao-financeira-2026-10-01.md.

## Contexto e premissas

Assinatura-base 49.90 (exemplo de .env.example, nao preco confirmado); Asaas 2.99% + 0.49 por recebimento, provisao 0.99 notificacao e 0.49 nota; tributos 6% e perdas 1% sao hipoteses; IA individual 5.00/assinante/mes. Custos fixos: 330 Render, 10 dominio, 60 ferramentas, 150 IA compartilhada/testes, 250 contabilidade, 3000 remuneracao total do fundador, 600 aquisicao.

Fontes consultadas em 2026-10-01: https://render.com/pricing ; https://www.asaas.com/precos-e-taxas ; ../../.env.example ; ../contracts/ai-cost-v1.md . Taxas contratuais e regime fiscal ainda nao verificados. Sem benchmark de capacidade do Render.


In [1]:
from decimal import Decimal, ROUND_CEILING, ROUND_HALF_UP
import json

def D(value):
    return Decimal(str(value))

def money(value):
    return float(D(value).quantize(D("0.01"), rounding=ROUND_HALF_UP))

def emit(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))

price = D("49.90")
gateway = D("0.0299")
tax = D("0.06")
loss = D("0.01")
per_payment = D("0.49") + D("0.99") + D("0.49")
ai = D("5")
fixed_items = {"render": 330, "domain": 10, "tools": 60, "platform_ai_trials": 150, "accounting": 250, "founder": 3000, "acquisition": 600}
fixed = sum(fixed_items.values())

def contribution(p=price, ai_cost=ai, tax_rate=tax):
    return D(p) * (1 - gateway - D(tax_rate) - loss) - per_payment - D(ai_cost)

def required(f=fixed, target=0, p=price, ai_cost=ai, tax_rate=tax):
    margin = contribution(p, ai_cost, tax_rate)
    return None if margin <= 0 else int(((D(f)+D(target))/margin).to_integral_value(rounding=ROUND_CEILING))

margin = contribution()
emit({"price":money(price),"card_fee":money(price*gateway+D("0.49")),"notification":0.99,"invoice":0.49,"tax":money(price*tax),"loss":money(price*loss),"ai":5,"contribution":money(margin),"contribution_percent":money(margin/price*100),"fixed":fixed,"fixed_items":fixed_items})


{
  "price": 49.9,
  "card_fee": 1.98,
  "notification": 0.99,
  "invoice": 0.49,
  "tax": 2.99,
  "loss": 0.5,
  "ai": 5,
  "contribution": 37.94,
  "contribution_percent": 76.04,
  "fixed": 4400,
  "fixed_items": {
    "render": 330,
    "domain": 10,
    "tools": 60,
    "platform_ai_trials": 150,
    "accounting": 250,
    "founder": 3000,
    "acquisition": 600
  }
}


## Pontos de equilibrio e resultados mensais

Assinantes sao cobrancas mensais pagas antes da provisao de estornos; usuarios gratuitos nao geram receita. Resultados incluem a provisao tributaria e remuneracao conforme cada escopo, sem afirmar lucro contabil definitivo.


In [2]:
breaks = {str(f): required(f=f) for f in [330,400,800,1400,3800,4400]}
goals = {str(target):required(target=target) for target in [0,1000,3000,5000]}
monthly = [{"subscribers":n,"revenue":money(price*n),"variable_cost":money((price-margin)*n),"contribution":money(margin*n),"result_before_founder":money(margin*n-(fixed-3000)),"result_after_founder":money(margin*n-fixed)} for n in [10,25,50,100,116,150,200,250]]
emit({"break_even_by_fixed_scope":breaks,"profit_targets":goals,"monthly":monthly})


{
  "break_even_by_fixed_scope": {
    "330": 9,
    "400": 11,
    "800": 22,
    "1400": 37,
    "3800": 101,
    "4400": 116
  },
  "profit_targets": {
    "0": 116,
    "1000": 143,
    "3000": 196,
    "5000": 248
  },
  "monthly": [
    {
      "subscribers": 10,
      "revenue": 499.0,
      "variable_cost": 119.55,
      "contribution": 379.45,
      "result_before_founder": -1020.55,
      "result_after_founder": -4020.55
    },
    {
      "subscribers": 25,
      "revenue": 1247.5,
      "variable_cost": 298.88,
      "contribution": 948.62,
      "result_before_founder": -451.38,
      "result_after_founder": -3451.38
    },
    {
      "subscribers": 50,
      "revenue": 2495.0,
      "variable_cost": 597.75,
      "contribution": 1897.25,
      "result_before_founder": 497.25,
      "result_after_founder": -2502.75
    },
    {
      "subscribers": 100,
      "revenue": 4990.0,
      "variable_cost": 1195.5,
      "contribution": 3794.5,
      "result_before_founder": 239

## Sensibilidades

Variacoes isoladas, sem assumir mesma conversao ou retencao em precos diferentes. A capacidade de hospedagem para essas bases ainda nao foi medida.


In [3]:
prices = [{"price":p,"margin":money(contribution(p)),"break_even":required(p=p),"target_1000":required(p=p,target=1000),"target_3000":required(p=p,target=3000),"target_5000":required(p=p,target=5000)} for p in [49.9,79.9,99.9]]
sensitivity = {"ai":[{"ai":v,"margin":money(contribution(ai_cost=v)),"break_even":required(ai_cost=v)} for v in [2,5,15,25]],"tax":[{"rate":v,"break_even":required(tax_rate=v)} for v in [0.06,0.10,0.155]],"infra_extra":[{"extra":v,"break_even":required(f=fixed+v)} for v in [0,100,300,600]]}
emit({"prices":prices,"sensitivity":sensitivity})


{
  "prices": [
    {
      "price": 49.9,
      "margin": 37.94,
      "break_even": 116,
      "target_1000": 143,
      "target_3000": 196,
      "target_5000": 248
    },
    {
      "price": 79.9,
      "margin": 64.95,
      "break_even": 68,
      "target_1000": 84,
      "target_3000": 114,
      "target_5000": 145
    },
    {
      "price": 99.9,
      "margin": 82.95,
      "break_even": 54,
      "target_1000": 66,
      "target_3000": 90,
      "target_5000": 114
    }
  ],
  "sensitivity": {
    "ai": [
      {
        "ai": 2,
        "margin": 40.94,
        "break_even": 108
      },
      {
        "ai": 5,
        "margin": 37.94,
        "break_even": 116
      },
      {
        "ai": 15,
        "margin": 27.94,
        "break_even": 158
      },
      {
        "ai": 25,
        "margin": 17.94,
        "break_even": 246
      }
    ],
    "tax": [
      {
        "rate": 0.06,
        "break_even": 116
      },
      {
        "rate": 0.1,
        "break_even": 

## Aquisicao e retencao

CAC em dinheiro nao inclui horas comerciais do fundador (ja orcadas como custo fixo). LTV simplificado e margem mensal/churn; nao e previsao de receita nem lucro liquido. A verba de 600 ja esta no fixo: nao descontar CAC novamente da DRE.


In [4]:
cac_rows = [{"cac":cac,"new_monthly":money(D(600)/D(cac)),"payback_months":money(D(cac)/margin),"steady_base_at_5pct_churn":money(D(600)/D(cac)/D(".05"))} for cac in [60,100,150,250]]
ltv_rows = [{"monthly_churn":c,"ltv_contribution":money(margin/D(c)),"ltv_over_cac_100":money(margin/D(c)/100),"replacement_budget_at_116_cac100":money(116*D(c)*100)} for c in [0.03,0.05,0.10]]
emit({"cac":cac_rows,"ltv":ltv_rows,"max_cac_for_three_month_payback":money(margin*3),"max_cac_for_steady_116_with_600_budget":money(D(600)/(116*D(".05"))),"new_for_net_10_at_116_churn_5pct":money(10+116*D(".05"))})


{
  "cac": [
    {
      "cac": 60,
      "new_monthly": 10.0,
      "payback_months": 1.58,
      "steady_base_at_5pct_churn": 200.0
    },
    {
      "cac": 100,
      "new_monthly": 6.0,
      "payback_months": 2.64,
      "steady_base_at_5pct_churn": 120.0
    },
    {
      "cac": 150,
      "new_monthly": 4.0,
      "payback_months": 3.95,
      "steady_base_at_5pct_churn": 80.0
    },
    {
      "cac": 250,
      "new_monthly": 2.4,
      "payback_months": 6.59,
      "steady_base_at_5pct_churn": 48.0
    }
  ],
  "ltv": [
    {
      "monthly_churn": 0.03,
      "ltv_contribution": 1264.83,
      "ltv_over_cac_100": 12.65,
      "replacement_budget_at_116_cac100": 348.0
    },
    {
      "monthly_churn": 0.05,
      "ltv_contribution": 758.9,
      "ltv_over_cac_100": 7.59,
      "replacement_budget_at_116_cac100": 580.0
    },
    {
      "monthly_churn": 0.1,
      "ltv_contribution": 379.45,
      "ltv_over_cac_100": 3.79,
      "replacement_budget_at_116_cac100": 1160.0


## Simulacao de 12 meses e caixa

Base inicial zero, 10 novos pagantes por mes, churn de 5% sobre a base anterior, CAC de caixa 60 e aquisicao mensal de 600. Novos pagam um mes cheio no mes de entrada; fracionarios sao valores esperados, nao pessoas medidas. Provisoes tratadas como desembolsos para prudencia. Recebimentos: cenario simples com atraso de um ciclo mensal, aproximacao de D+32; nao e fluxo bancario diario. Setup hipotetico de 1500 no mes zero. Sem crescimento de custos fixos.


In [5]:
base=D(0)
cumulative=D(0)
cash=D(-1500)
previous_net_receipt=D(0)
rows=[]
net_receipt_per_user=price*(1-gateway-tax-loss)-per_payment
for month in range(1,13):
    base=base*D(".95")+10
    revenue=base*price
    result=base*margin-fixed
    cumulative+=result
    cash+=previous_net_receipt-base*ai-fixed
    previous_net_receipt=base*net_receipt_per_user
    rows.append({"month":month,"expected_payers":money(base),"revenue":money(revenue),"monthly_result":money(result),"cumulative_result":money(cumulative),"cash_after_setup_and_receipt_lag":money(cash)})
emit({"months":rows,"annual_revenue":money(sum(D(r["revenue"]) for r in rows)),"annual_result":money(cumulative),"annual_result_without_founder":money(cumulative+36000),"max_cash_need_first_12_months":money(-min(D(r["cash_after_setup_and_receipt_lag"]) for r in rows)),"reserve_with_20pct_buffer":money(-min(D(r["cash_after_setup_and_receipt_lag"]) for r in rows)*D("1.2"))})
base=D(0)
for month in range(1,121):
    base=base*D(".95")+10
    if base*margin>=fixed:
        emit({"first_monthly_break_even_under_same_assumptions":month,"expected_payers":money(base)})
        break


{
  "months": [
    {
      "month": 1,
      "expected_payers": 10.0,
      "revenue": 499.0,
      "monthly_result": -4020.55,
      "cumulative_result": -4020.55,
      "cash_after_setup_and_receipt_lag": -5950.0
    },
    {
      "month": 2,
      "expected_payers": 19.5,
      "revenue": 973.05,
      "monthly_result": -3660.07,
      "cumulative_result": -7680.62,
      "cash_after_setup_and_receipt_lag": -10018.05
    },
    {
      "month": 3,
      "expected_payers": 28.53,
      "revenue": 1423.4,
      "monthly_result": -3317.62,
      "cumulative_result": -10998.24,
      "cash_after_setup_and_receipt_lag": -13723.25
    },
    {
      "month": 4,
      "expected_payers": 37.1,
      "revenue": 1851.23,
      "monthly_result": -2992.29,
      "cumulative_result": -13990.53,
      "cash_after_setup_and_receipt_lag": -17083.74
    },
    {
      "month": 5,
      "expected_payers": 45.24,
      "revenue": 2257.67,
      "monthly_result": -2683.22,
      "cumulative_result": 

## Validacao aritmetica

Testes de fronteira confirmam que N cobre o objetivo e N-1 nao cobre. A identidade receita menos custos foi recalculada diretamente. Estes testes validam contas, nao as premissas comerciais.


In [6]:
cash_without_founder = [D(row["cash_after_setup_and_receipt_lag"]) + D(3000)*row["month"] for row in rows]
need_without_founder = -min(cash_without_founder)
emit({"cash_need_without_founder":money(need_without_founder),"buffered_cash_need_without_founder":money(need_without_founder*D("1.2")),"price_break_even_with_100_payers":money((D(4400)/100+per_payment+ai)/(1-gateway-tax-loss))})


{
  "cash_need_without_founder": 5116.75,
  "buffered_cash_need_without_founder": 6140.1,
  "price_break_even_with_100_payers": 56.63
}


In [7]:
assert fixed == 4400
assert margin == D("37.94499")
for p in [49.9,79.9,99.9]:
    for target in [0,1000,3000,5000]:
        n=required(p=p,target=target)
        c=contribution(p)
        assert n*c-fixed >= target
        assert (n-1)*c-fixed < target
for row in monthly:
    n=row["subscribers"]
    direct = n*price - n*price*gateway - n*per_payment - n*price*tax - n*price*loss - n*ai - fixed
    assert money(direct)==row["result_after_founder"]
assert required()==116
assert required(p=79.9)==68
assert required(p=99.9)==54
print("Contas e limites de equilibrio validados. Premissas de mercado e capacidade ainda nao validadas.")


Contas e limites de equilibrio validados. Premissas de mercado e capacidade ainda nao validadas.
